In [0]:
CREATE OR REPLACE PROCEDURE moh_raz.raz_casemix_stg.SP_CAPTURE_DELETIONS
(
    IN p_catalog_name     STRING,
    IN p_schema_name      STRING,
    IN p_table_name       STRING,
    IN p_capture_columns  STRING
)
LANGUAGE SQL
SQL SECURITY INVOKER
MODIFIES SQL DATA
AS
BEGIN
    -- VARIABLES
    DECLARE v_last_version     BIGINT DEFAULT 0;
    DECLARE v_start_version    BIGINT DEFAULT 1;
    DECLARE v_current_version  BIGINT DEFAULT 1;
    DECLARE v_source_table     STRING;
    DECLARE v_capture_expr     STRING;
    DECLARE v_sql              STRING;

    -- Create Control and Audit Tables
    CREATE TABLE IF NOT EXISTS moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT_CONTROL
        (
            catalog_name            STRING,
            schema_name             STRING,
            table_name              STRING,
            capture_columns         STRING,
            last_processed_version  BIGINT,
            is_active               INT,
            updated_at              TIMESTAMP
        )
        USING DELTA;

    CREATE TABLE IF NOT EXISTS moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT
        (
            audit_id          STRING,
            catalog_name      STRING,
            schema_name       STRING,
            table_name        STRING,
            primary_id        STRING,
            deleted_by        STRING,
            deleted_at        TIMESTAMP,
            commit_version    BIGINT,
            commit_timestamp  TIMESTAMP,
            operation         STRING,
            loaded_at         TIMESTAMP
        )
        USING DELTA;
    
    -- 1. Build fully qualified source table
    SET v_source_table = CONCAT(
        '`',
        p_catalog_name,
        '`.`',
        p_schema_name,
        '`.`',
        p_table_name,
        '`'
    );

    -- 2. Get last processed version
    SET v_last_version = COALESCE(
        (
            SELECT MAX(last_processed_version)
            FROM moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT_CONTROL
            WHERE catalog_name = p_catalog_name
              AND schema_name = p_schema_name
              AND table_name = p_table_name
              AND is_active = 1
        ),
        0
    );

    -- 3. Calculate next version to process
    SET v_start_version = v_last_version + 1;

    -- 4. Get current/latest Delta version
    SET v_sql = CONCAT(
        'SELECT COALESCE(MAX(version), -1) ',
        'FROM (DESCRIBE HISTORY ',
        v_source_table,
        ')'
    );


    EXECUTE IMMEDIATE v_sql
    INTO v_current_version;


    
    -- 5. Process only when new versions are available
    IF v_current_version >= v_start_version THEN

        -- 6. Build primary_id expression    
        SET v_capture_expr =
            CASE

                WHEN SIZE(SPLIT(p_capture_columns, ',')) = 1

                THEN
                    CONCAT(
                        'CAST(`',
                        TRIM(p_capture_columns),
                        '` AS STRING)'
                    )

                ELSE
                    CONCAT(
                        'concat_ws(''|'', ',
                        ARRAY_JOIN(
                            TRANSFORM(
                                SPLIT(p_capture_columns, ','),
                                x -> CONCAT(
                                    'CAST(`',
                                    TRIM(x),
                                    '` AS STRING)'
                                )
                            ),
                            ', '
                        ),
                        ')'
                    )

            END;


        
        -- 7. Build DELETE audit INSERT
        SET v_sql = CONCAT(

            'INSERT INTO ',
            'moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT',
            
            '(',
            'audit_id, ',
            'catalog_name, ',
            'schema_name, ',
            'table_name, ',
            'primary_id, ',
            'deleted_by, ',
            'deleted_at, ',
            'commit_version, ',
            'commit_timestamp, ',
            'operation, ',
            'loaded_at',
            ') ',

            'SELECT ',

            'uuid(), ',

            CHR(39),
            p_catalog_name,
            CHR(39),
            ', ',

            CHR(39),
            p_schema_name,
            CHR(39),
            ', ',

            CHR(39),
            p_table_name,
            CHR(39),
            ', ',

            v_capture_expr,
            ', ',

            'h.userName, ',

            'c._commit_timestamp, ',

            'c._commit_version, ',

            'c._commit_timestamp, ',

            CHR(39),
            'DELETE',
            CHR(39),
            ', ',

            'current_timestamp() ',

            'FROM table_changes(',

            CHR(39),
            v_source_table,
            CHR(39),

            ', ',

            CAST(v_start_version AS STRING),

            ', ',

            CAST(v_current_version AS STRING),

            ') c ',

            'LEFT JOIN ',

            '(DESCRIBE HISTORY ',
            v_source_table,
            ') h ',

            'ON c._commit_version = h.version ',

            'WHERE c._change_type = ',

            CHR(39),
            'delete',
            CHR(39)

        );


        
        -- 8. Execute DELETE audit INSERT
        EXECUTE IMMEDIATE v_sql;

        -- 9. Update control table
        MERGE INTO
            moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT_CONTROL AS tgt

        USING
        (
            SELECT
                p_catalog_name AS catalog_name,
                p_schema_name AS schema_name,
                p_table_name AS table_name,
                p_capture_columns AS capture_columns,
                v_current_version AS last_processed_version

        ) AS src

        ON  tgt.catalog_name = src.catalog_name
        AND tgt.schema_name = src.schema_name
        AND tgt.table_name = src.table_name

        WHEN MATCHED THEN

            UPDATE SET

                tgt.capture_columns =
                    src.capture_columns,

                tgt.last_processed_version =
                    src.last_processed_version,

                tgt.updated_at =
                    current_timestamp()

        WHEN NOT MATCHED THEN

            INSERT
            (
                catalog_name,
                schema_name,
                table_name,
                capture_columns,
                last_processed_version,
                is_active,
                updated_at
            )

            VALUES
            (
                src.catalog_name,
                src.schema_name,
                src.table_name,
                src.capture_columns,
                src.last_processed_version,
                1,
                current_timestamp()
            );

    END IF;

END;